# 06 — Pipeline end-to-end: Product → Issue

## Objetivo

En los notebooks anteriores se evaluaron por separado dos niveles de clasificación: **Product** e **Issue**. En `Issue` se utilizó el Product real para aislar el rendimiento de la segunda etapa.

En este notebook se construye y evalúa el pipeline completo:

**Narrativa → Product predicho → Issue predicho**

La evaluación se realizará sobre un *holdout* común no utilizado para entrenar ninguno de los dos modelos, evitando *data leakage*.

### Objetivos
- Reconstruir los conjuntos de Product e Issue.
- Crear un holdout común.
- Evaluar Product.
- Evaluar Issue con Product predicho.
- Comparar con el escenario *oracle* usando Product real.
- Cuantificar la propagación de errores.
- Crear una función final de inferencia.

In [1]:
from pathlib import Path
import pickle
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
FECHA_INICIO = pd.Timestamp("2023-09-01")
MIN_COMBINACION = 20

ROOT = Path("..")
DATA_RAW = ROOT / "data" / "raw"
MODELS_DIR = ROOT / "models"
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PRODUCT_VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"
PRODUCT_MODEL_PATH = MODELS_DIR / "linear_svm_model.pkl"
ISSUE_MODEL_PATH = MODELS_DIR / "issue_hierarchical_svm.pkl"

print("DATA_RAW:", DATA_RAW.resolve())
print("MODELS_DIR:", MODELS_DIR.resolve())

DATA_RAW: C:\Users\Usuario\Documents\UCM Master\TFM Ana Valeria\data\raw
MODELS_DIR: C:\Users\Usuario\Documents\UCM Master\TFM Ana Valeria\models


## 1. Localización del fichero CFPB

Se busca automáticamente en `data/raw` un CSV que contenga las columnas necesarias.

In [2]:
COLUMNAS_NECESARIAS = {
    "Date received",
    "Consumer complaint narrative",
    "Product",
    "Issue"
}

def localizar_cfpb(data_dir):
    for archivo in data_dir.glob("*.csv"):
        try:
            cols = set(pd.read_csv(archivo, nrows=0).columns)
            if COLUMNAS_NECESARIAS.issubset(cols):
                return archivo
        except Exception:
            pass
    raise FileNotFoundError("No se encontró el CSV del CFPB en data/raw.")

CFPB_PATH = localizar_cfpb(DATA_RAW)
print("Fichero CFPB:", CFPB_PATH.name)

Fichero CFPB: complaints.csv


## 2. Lectura del periodo estable

Se mantiene la misma decisión metodológica: reclamaciones recibidas desde el **1 de septiembre de 2023**. El fichero se procesa por bloques para reducir el uso de memoria.

In [3]:
USECOLS = ["Date received", "Consumer complaint narrative", "Product", "Issue"]
chunks_filtrados = []
t0 = time.time()

for chunk in pd.read_csv(CFPB_PATH, usecols=USECOLS, chunksize=500_000, low_memory=False):
    chunk["Date received"] = pd.to_datetime(chunk["Date received"], errors="coerce")
    chunk = chunk.loc[chunk["Date received"] >= FECHA_INICIO].copy()
    if len(chunk):
        chunks_filtrados.append(chunk)

df_periodo = pd.concat(chunks_filtrados, ignore_index=True)
print(f"Registros desde {FECHA_INICIO.date()}: {len(df_periodo):,}")
print(f"Tiempo: {(time.time()-t0)/60:.2f} min")

Registros desde 2023-09-01: 12,982,646
Tiempo: 0.92 min


## 3. Reconstrucción del corpus de Product

Se eliminan narrativas nulas o vacías, narrativas asociadas a más de un Product y duplicados por narrativa.

In [4]:
COL_TEXTO = "Consumer complaint narrative"

df_product = df_periodo[["Date received", COL_TEXTO, "Product"]].copy()
df_product = df_product.dropna(subset=[COL_TEXTO, "Product"]).copy()
df_product[COL_TEXTO] = df_product[COL_TEXTO].astype(str).str.strip()
df_product = df_product[df_product[COL_TEXTO] != ""].copy()

n_prod = df_product.groupby(COL_TEXTO)["Product"].nunique()
ambiguos_product = set(n_prod[n_prod > 1].index)

df_product_clean = (
    df_product[~df_product[COL_TEXTO].isin(ambiguos_product)]
    .drop_duplicates(subset=[COL_TEXTO], keep="first")
    .reset_index(drop=True)
)

print(f"Corpus Product: {len(df_product_clean):,}")
print(f"Products: {df_product_clean['Product'].nunique()}")

Corpus Product: 1,299,353
Products: 11


## 4. Reconstrucción del corpus de Issue

Se eliminan ambigüedades de Product e Issue, se deduplica por narrativa y se conservan combinaciones Product–Issue con al menos 20 observaciones.

In [5]:
df_issue = df_periodo[["Date received", COL_TEXTO, "Product", "Issue"]].copy()
df_issue = df_issue.dropna(subset=[COL_TEXTO, "Product", "Issue"]).copy()
df_issue[COL_TEXTO] = df_issue[COL_TEXTO].astype(str).str.strip()
df_issue = df_issue[df_issue[COL_TEXTO] != ""].copy()

n_prod_i = df_issue.groupby(COL_TEXTO)["Product"].nunique()
n_issue = df_issue.groupby(COL_TEXTO)["Issue"].nunique()
amb_prod = set(n_prod_i[n_prod_i > 1].index)
amb_issue = set(n_issue[n_issue > 1].index)

df_issue_clean = (
    df_issue[~df_issue[COL_TEXTO].isin(amb_prod | amb_issue)]
    .drop_duplicates(subset=[COL_TEXTO], keep="first")
    .reset_index(drop=True)
)

df_issue_clean["Product_Issue"] = df_issue_clean["Product"] + " || " + df_issue_clean["Issue"]
freq = df_issue_clean["Product_Issue"].value_counts()
validas = set(freq[freq >= MIN_COMBINACION].index)

df_issue_model = df_issue_clean[df_issue_clean["Product_Issue"].isin(validas)].copy().reset_index(drop=True)

print(f"Dataset Issue limpio:   {len(df_issue_clean):,}")
print(f"Dataset modelado:       {len(df_issue_model):,}")
print(f"Products:               {df_issue_model['Product'].nunique()}")
print(f"Issues:                 {df_issue_model['Issue'].nunique()}")
print(f"Combinaciones:          {df_issue_model['Product_Issue'].nunique()}")

Dataset Issue limpio:   1,295,164
Dataset modelado:       1,295,104
Products:               11
Issues:                 86
Combinaciones:          128


## 5. Holdout común sin fuga de información

El test de Product y el test de Issue no son idénticos. Para una evaluación end-to-end rigurosa se toma la **intersección** de ambos tests, de modo que cada narrativa haya sido reservada como test en las dos tareas.

In [6]:
_, product_test = train_test_split(
    df_product_clean,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df_product_clean["Product"]
)

_, issue_test = train_test_split(
    df_issue_model,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df_issue_model["Product_Issue"]
)

textos_holdout = set(product_test[COL_TEXTO]) & set(issue_test[COL_TEXTO])
holdout = issue_test[issue_test[COL_TEXTO].isin(textos_holdout)].copy().reset_index(drop=True)

print(f"Test Product:             {len(product_test):,}")
print(f"Test Issue:               {len(issue_test):,}")
print(f"Holdout común end-to-end: {len(holdout):,}")
print(f"Products en holdout:      {holdout['Product'].nunique()}")
print(f"Issues en holdout:        {holdout['Issue'].nunique()}")

Test Product:             259,871
Test Issue:               259,021
Holdout común end-to-end: 51,777
Products en holdout:      11
Issues en holdout:        85


## 6. Carga de modelos guardados

Se utiliza DistilBERT para la clasificación de Product y un modelo jerárquico basado en TF-IDF + Linear SVM para la clasificación de Issue. La categoría Product predicha por DistilBERT determina qué modelo específico se utiliza en la segunda etapa.

In [20]:
import joblib

# Carga del modelo jerárquico de Issue

if not ISSUE_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró: {ISSUE_MODEL_PATH.resolve()}"
    )

artefacto_issue = joblib.load(ISSUE_MODEL_PATH)

modelos_issue = artefacto_issue["modelos_por_producto"]

print("Modelo jerárquico de Issue cargado correctamente.")
print("Modelos Issue por Product:", len(modelos_issue))

Modelo jerárquico de Issue cargado correctamente.
Modelos Issue por Product: 11


In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DISTILBERT_PATH = MODELS_DIR / "distilbert_cfpb_250k"

# Tokenizer del modelo base
tokenizer_product = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

# Nuestro modelo fine-tuned
model_product_transformer = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_PATH
)

print("DistilBERT cargado correctamente.")
print("Número de clases:", model_product_transformer.config.num_labels)
print("id2label:", model_product_transformer.config.id2label)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT cargado correctamente.
Número de clases: 11
id2label: {0: 'Checking or savings account', 1: 'Credit card', 2: 'Credit reporting or other personal consumer reports', 3: 'Debt collection', 4: 'Debt or credit management', 5: 'Money transfer, virtual currency, or money service', 6: 'Mortgage', 7: 'Payday loan, title loan, personal loan, or advance loan', 8: 'Prepaid card', 9: 'Student loan', 10: 'Vehicle loan or lease'}


In [22]:
from sklearn.model_selection import train_test_split
import pandas as pd

SEED = 42

ruta_250k = PROCESSED_DIR / "complaints_transformer_250k.csv"

df_transformer_250k = pd.read_csv(
    ruta_250k,
    low_memory=False
)

TEXT_COL = "Consumer complaint narrative"
TARGET_COL = "Product"

X = df_transformer_250k[TEXT_COL]
y = df_transformer_250k[TARGET_COL]

X_train_250k, X_temp_250k, y_train_250k, y_temp_250k = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_val_250k, X_test_250k, y_val_250k, y_test_250k = train_test_split(
    X_temp_250k,
    y_temp_250k,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp_250k
)

print("Train:", len(X_train_250k))
print("Validation:", len(X_val_250k))
print("Test:", len(X_test_250k))
print("Products en test:", y_test_250k.nunique())

Train: 200000
Validation: 25000
Test: 25000
Products en test: 11


In [23]:
# Construcción del holdout común:
# test de DistilBERT ∩ test de Issue

df_test_transformer = pd.DataFrame({
    TEXT_COL: X_test_250k.values,
    "Product": y_test_250k.values
})

textos_test_transformer = set(df_test_transformer[TEXT_COL])
textos_test_issue = set(issue_test[COL_TEXTO])

textos_holdout_final = textos_test_transformer & textos_test_issue

holdout_final = issue_test[
    issue_test[COL_TEXTO].isin(textos_holdout_final)
].copy()

print("Test DistilBERT:", len(df_test_transformer))
print("Test Issue:", len(issue_test))
print("Holdout final end-to-end:", len(holdout_final))
print("Products en holdout:", holdout_final["Product"].nunique())
print("Issues en holdout:", holdout_final["Issue"].nunique())

Test DistilBERT: 25000
Test Issue: 259021
Holdout final end-to-end: 5086
Products en holdout: 11
Issues en holdout: 75


## 7. Predicción de Product con DistilBERT

Se utiliza el modelo DistilBERT seleccionado en la fase de modelado para predecir la categoría `Product` sobre el holdout común. Estas predicciones constituirán la primera etapa del pipeline jerárquico end-to-end.

In [24]:
import torch
from tqdm.auto import tqdm

model_product_transformer.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_product_transformer.to(device)

print("Dispositivo:", device)

textos = holdout_final[COL_TEXTO].tolist()

predicciones_product = []

BATCH_SIZE = 16

for i in tqdm(range(0, len(textos), BATCH_SIZE)):
    
    batch_textos = textos[i:i + BATCH_SIZE]

    inputs = tokenizer_product(
        batch_textos,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model_product_transformer(**inputs)

    pred_ids = torch.argmax(
        outputs.logits,
        dim=1
    ).cpu().numpy()

    pred_labels = [
        model_product_transformer.config.id2label[int(idx)]
        for idx in pred_ids
    ]

    predicciones_product.extend(pred_labels)

holdout_final["Product_pred"] = predicciones_product

print("\nPredicciones realizadas:", len(predicciones_product))

Dispositivo: cpu


  0%|          | 0/318 [00:00<?, ?it/s]


Predicciones realizadas: 5086


In [25]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_true_product = holdout_final["Product"]
y_pred_product = holdout_final["Product_pred"]

accuracy_product = accuracy_score(
    y_true_product,
    y_pred_product
)

f1_macro_product = f1_score(
    y_true_product,
    y_pred_product,
    average="macro"
)

f1_weighted_product = f1_score(
    y_true_product,
    y_pred_product,
    average="weighted"
)

print("=== PRODUCT - DISTILBERT EN HOLDOUT COMÚN ===")
print(f"Accuracy:    {accuracy_product:.4f}")
print(f"F1 macro:    {f1_macro_product:.4f}")
print(f"F1 weighted: {f1_weighted_product:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_true_product,
        y_pred_product,
        digits=4
    )
)

=== PRODUCT - DISTILBERT EN HOLDOUT COMÚN ===
Accuracy:    0.8748
F1 macro:    0.7594
F1 weighted: 0.8725

Classification report:
                                                         precision    recall  f1-score   support

                            Checking or savings account     0.8089    0.8107    0.8098       428
                                            Credit card     0.8371    0.7591    0.7962       386
    Credit reporting or other personal consumer reports     0.9201    0.9563    0.9379      2952
                                        Debt collection     0.7820    0.7398    0.7603       611
                              Debt or credit management     0.7143    0.3333    0.4545        15
     Money transfer, virtual currency, or money service     0.7511    0.7607    0.7558       234
                                               Mortgage     0.8913    0.8978    0.8945       137
Payday loan, title loan, personal loan, or advance loan     0.7167    0.5972    0.6515       

## 9. Funciones auxiliares de Issue

In [28]:
def resolver_componentes_issue(entrada):
    if isinstance(entrada, dict):
        vec_keys = ["vectorizer", "vectorizador", "tfidf", "tfidf_vectorizer"]
        mod_keys = ["model", "modelo", "classifier", "clasificador", "svm"]
        vec = next((entrada[k] for k in vec_keys if k in entrada), None)
        mod = next((entrada[k] for k in mod_keys if k in entrada), None)
        if vec is not None and mod is not None:
            return vec, mod
    if isinstance(entrada, (tuple, list)) and len(entrada) >= 2:
        return entrada[0], entrada[1]
    raise TypeError(f"Estructura de modelo Issue no reconocida: {type(entrada)}")

def predecir_issue_por_producto(textos, productos):
    textos = pd.Series(textos).reset_index(drop=True)
    productos = pd.Series(productos).reset_index(drop=True)
    pred = np.empty(len(textos), dtype=object)

    for producto in productos.unique():
        idx = np.where(productos.values == producto)[0]
        if producto not in modelos_issue:
            pred[idx] = None
            continue

        entrada = modelos_issue[producto]
        if isinstance(entrada, dict) and entrada.get("tipo") == "constante":
            valor = entrada.get("clase") or entrada.get("issue") or entrada.get("valor")
            pred[idx] = valor
            continue

        vec, mod = resolver_componentes_issue(entrada)
        X = vec.transform(textos.iloc[idx])
        pred[idx] = mod.predict(X)

    return pred

In [29]:
# Predicción end-to-end de Issue usando el Product predicho por DistilBERT

holdout_final["Issue_pred"] = predecir_issue_por_producto(
    holdout_final[COL_TEXTO],
    holdout_final["Product_pred"]
)

print("Predicciones Issue realizadas:", holdout_final["Issue_pred"].notna().sum())
print("Valores nulos:", holdout_final["Issue_pred"].isna().sum())

Predicciones Issue realizadas: 5086
Valores nulos: 0


## 10. Evaluación end-to-end de Issue

Se utiliza el **Product predicho** para seleccionar el modelo de Issue correspondiente.

In [30]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_true_issue = holdout_final["Issue"]
y_pred_issue = holdout_final["Issue_pred"]

accuracy_issue_e2e = accuracy_score(
    y_true_issue,
    y_pred_issue
)

f1_macro_issue_e2e = f1_score(
    y_true_issue,
    y_pred_issue,
    average="macro"
)

f1_weighted_issue_e2e = f1_score(
    y_true_issue,
    y_pred_issue,
    average="weighted"
)

print("=== ISSUE - PIPELINE END-TO-END ===")
print(f"Accuracy:    {accuracy_issue_e2e:.4f}")
print(f"F1 macro:    {f1_macro_issue_e2e:.4f}")
print(f"F1 weighted: {f1_weighted_issue_e2e:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_true_issue,
        y_pred_issue,
        digits=4,
        zero_division=0
    )
)

=== ISSUE - PIPELINE END-TO-END ===
Accuracy:    0.6258
F1 macro:    0.3745
F1 weighted: 0.6157

Classification report:
                                                                 precision    recall  f1-score   support

                                                    Advertising     1.0000    1.0000    1.0000         1
        Advertising and marketing, including promotional offers     0.6316    0.4800    0.5455        25
    Applying for a mortgage or refinancing an existing mortgage     0.6111    0.5789    0.5946        19
                              Attempts to collect debt not owed     0.5096    0.5882    0.5461       272
                               Can't contact lender or servicer     0.0000    0.0000    0.0000         3
                  Can't stop withdrawals from your bank account     0.0000    0.0000    0.0000         1
                     Charged fees or interest you didn't expect     0.3478    0.5333    0.4211        15
                             Charged up

## 11. Escenario oracle

Se repite la predicción de Issue utilizando el **Product real** para cuantificar cuánto rendimiento se pierde por la propagación de errores.

In [31]:
holdout_final["Issue_pred_oracle"] = predecir_issue_por_producto(
    holdout_final[COL_TEXTO],
    holdout_final["Product"]
)

accuracy_issue_oracle = accuracy_score(
    holdout_final["Issue"],
    holdout_final["Issue_pred_oracle"]
)

f1_macro_issue_oracle = f1_score(
    holdout_final["Issue"],
    holdout_final["Issue_pred_oracle"],
    average="macro",
    zero_division=0
)

f1_weighted_issue_oracle = f1_score(
    holdout_final["Issue"],
    holdout_final["Issue_pred_oracle"],
    average="weighted",
    zero_division=0
)

print("=== ISSUE - ESCENARIO ORACLE ===")
print(f"Accuracy:    {accuracy_issue_oracle:.4f}")
print(f"F1 macro:    {f1_macro_issue_oracle:.4f}")
print(f"F1 weighted: {f1_weighted_issue_oracle:.4f}")

=== ISSUE - ESCENARIO ORACLE ===
Accuracy:    0.6897
F1 macro:    0.4801
F1 weighted: 0.6803


## 12. Comparación oracle vs end-to-end

In [32]:
comparacion = pd.DataFrame({
    "Escenario": [
        "Oracle (Product real)",
        "End-to-end (Product predicho)"
    ],
    "Accuracy": [
        accuracy_issue_oracle,
        accuracy_issue_e2e
    ],
    "F1 macro": [
        f1_macro_issue_oracle,
        f1_macro_issue_e2e
    ],
    "F1 weighted": [
        f1_weighted_issue_oracle,
        f1_weighted_issue_e2e
    ]
})

comparacion

,Escenario,Accuracy,F1 macro,F1 weighted
0,Oracle (Product real),0.689737,0.480088,0.680286
1,End-to-end (Product predicho),0.625836,0.374548,0.615674


In [33]:
caida_macro = f1_macro_issue_oracle - f1_macro_issue_e2e

caida_relativa = (
    caida_macro / f1_macro_issue_oracle * 100
)

print(f"Caída absoluta F1 macro: {caida_macro:.4f}")
print(f"Caída relativa F1 macro: {caida_relativa:.2f}%")

Caída absoluta F1 macro: 0.1055
Caída relativa F1 macro: 21.98%


## 13. Propagación de errores

Se compara el rendimiento de Issue cuando Product fue clasificado correctamente y cuando fue clasificado incorrectamente.

In [34]:
# Identificamos si DistilBERT acertó el Product

holdout_final["Product_correcto"] = (
    holdout_final["Product"] == holdout_final["Product_pred"]
)

correctos = holdout_final[
    holdout_final["Product_correcto"]
]

incorrectos = holdout_final[
    ~holdout_final["Product_correcto"]
]

print("=== PROPAGACIÓN DEL ERROR ===")

print(f"Total holdout: {len(holdout_final)}")

print(
    f"Product correcto: {len(correctos)} "
    f"({len(correctos)/len(holdout_final):.2%})"
)

print(
    f"Product incorrecto: {len(incorrectos)} "
    f"({len(incorrectos)/len(holdout_final):.2%})"
)

print("\n--- ISSUE CUANDO PRODUCT ES CORRECTO ---")

print(
    f"Accuracy: "
    f"{accuracy_score(correctos['Issue'], correctos['Issue_pred']):.4f}"
)

print(
    f"F1 macro: "
    f"{f1_score(correctos['Issue'], correctos['Issue_pred'], average='macro', zero_division=0):.4f}"
)

print("\n--- ISSUE CUANDO PRODUCT ES INCORRECTO ---")

print(
    f"Accuracy: "
    f"{accuracy_score(incorrectos['Issue'], incorrectos['Issue_pred']):.4f}"
)

print(
    f"F1 macro: "
    f"{f1_score(incorrectos['Issue'], incorrectos['Issue_pred'], average='macro', zero_division=0):.4f}"
)

=== PROPAGACIÓN DEL ERROR ===
Total holdout: 5086
Product correcto: 4449 (87.48%)
Product incorrecto: 637 (12.52%)

--- ISSUE CUANDO PRODUCT ES CORRECTO ---
Accuracy: 0.7080
F1 macro: 0.4888

--- ISSUE CUANDO PRODUCT ES INCORRECTO ---
Accuracy: 0.0518
F1 macro: 0.0257


### Interpretación

Los resultados muestran una fuerte dependencia entre las dos etapas del pipeline jerárquico. DistilBERT clasifica correctamente el Product en el 87,48 % de las observaciones del holdout. Cuando esta primera clasificación es correcta, el modelo de Issue alcanza una accuracy de 0,7080 y un F1 macro de 0,4888.

Sin embargo, cuando Product se clasifica incorrectamente, el rendimiento de Issue disminuye hasta una accuracy de 0,0518 y un F1 macro de 0,0257. Esto se debe a que el Product predicho determina qué clasificador específico de Issue recibe la narrativa. Por tanto, un error en la primera etapa puede dirigir la observación hacia un espacio de clases que no corresponde con su Product real.

Estos resultados evidencian la propagación de errores característica de una arquitectura jerárquica y explican la diferencia observada entre el escenario oracle y la evaluación end-to-end.

## 14. Ejemplos de errores end-to-end

In [35]:
errores = holdout_final[
    (holdout_final["Product"] != holdout_final["Product_pred"]) |
    (holdout_final["Issue"] != holdout_final["Issue_pred"])
].copy()

pd.set_option("display.max_colwidth", 250)

errores[
    [COL_TEXTO, "Product", "Product_pred", "Issue", "Issue_pred"]
].sample(
    n=min(15, len(errores)),
    random_state=RANDOM_STATE
)

,Consumer complaint narrative,Product,Product_pred,Issue,Issue_pred
311023,"I had the checking and saving account in Chase Bank, and this account is used to pay all of my fees, including the housing rent, the tuition, and all other daily cost. However, last week my account has been closed and I can't access the money in ...",Checking or savings account,Checking or savings account,Managing an account,Closing an account
674995,I was looking at my report and notice there were some things that was not correct. You never reached out to the creditors and they are still on my report.I made a dispute to XXXX on XXXX XXXX and I called in and stated none of these accounts on m...,Debt collection,Credit reporting or other personal consumer reports,Attempts to collect debt not owed,Incorrect information on your report
16261,I paid this debt years ago To : XXXX XXXXXXXX XXXXXXXX Date : XX/XX/year> Account No : XXXX Amount : {$270.00},Debt collection,Debt collection,False statements or representation,Attempts to collect debt not owed
289189,"I sent a letter to the Credit Bureaus requesting to reinvestigate the disputed Account from my credit report as well as DELETE the Account from my report during the investigation period. As of this date, they have failed to respond to my request....",Credit reporting or other personal consumer reports,Credit reporting or other personal consumer reports,Incorrect information on your report,Problem with a company's investigation into an existing problem
50339,Information is reported inaccurately violating my rights as a consumer,Student loan,Credit reporting or other personal consumer reports,Incorrect information on your report,Improper use of your report
261503,"XXXX has put inaccurate Fraudulent information on XXXX, Experian XXXX credit reports and have not removed them after I disputed them countless times I never agreed to pay them anything I never opened up an account with them and I never used their...",Credit reporting or other personal consumer reports,Credit reporting or other personal consumer reports,Problem with a company's investigation into an existing problem,Incorrect information on your report
544422,"I have lost $ XXXX in financial scam perpetrated by XXXX and their actors. Out of $ XXXX, I lost approx $ XXXX in wire transfer fraud perpetrated by two entities XXXX XXXX and XXXX XXXX XXXX XXXX The remaining $ XXXX I lost in crypto funds. \n\nI...",Checking or savings account,"Money transfer, virtual currency, or money service",Managing an account,Fraud or scam
960295,"To Whom It May Concern, I am submitting this formal complaint regarding multiple inaccurate and harmful entries being reported on my credit files across the major credit reporting agencies ( Experian, Equifax, and TransUnion ). I have previously ...",Credit reporting or other personal consumer reports,Credit reporting or other personal consumer reports,Problem with a company's investigation into an existing problem,Improper use of your report
562664,Unauthorized calls to family members asking about acct opening of credit cards. Informed that another acct was opened XXXX different branches XXXX being a credit card. Etc. they have my family members phone number who is not listed on any of my a...,Checking or savings account,Credit card,Closing an account,Getting a credit card
591271,"Dear CFPB, I am writing to file a formal complaint against the major credit bureaus for their repeated failure to comply with the provisions of the Fair Credit Reporting Act ( FCRA ), as well as other applicable consumer protection laws, in respo...",Credit reporting or other personal consumer reports,Credit reporting or other personal consumer reports,Problem with a company's investigation into an existing problem,Improper use of your report


## 15. Función final de inferencia

Esta función encapsula el pipeline completo y será reutilizable posteriormente desde `src/` y desde la aplicación.

In [36]:
import torch

def predecir_pipeline(texto):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("El texto debe ser una cadena no vacía.")

    texto = texto.strip()

    # 1. Predicción de Product con DistilBERT
    inputs = tokenizer_product(
        [texto],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    device = next(model_product_transformer.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model_product_transformer.eval()

    with torch.no_grad():
        outputs = model_product_transformer(**inputs)

    pred_id = torch.argmax(outputs.logits, dim=1).item()

    product_pred = model_product_transformer.config.id2label[pred_id]

    # 2. Predicción de Issue con el modelo específico del Product
    issue_pred = predecir_issue_por_producto(
        [texto],
        [product_pred]
    )[0]

    return {
        "Product": product_pred,
        "Issue": issue_pred
    }

In [37]:
predecir_pipeline(
    "There is an account on my credit report that does not belong to me and I already disputed it."
)

{'Product': 'Credit reporting or other personal consumer reports',
 'Issue': 'Incorrect information on your report'}

## 16. Conclusiones

En este notebook se ha construido y evaluado el pipeline jerárquico completo para la clasificación automática de reclamaciones del CFPB. La arquitectura final combina **DistilBERT para la predicción de Product** con clasificadores **TF-IDF + Linear SVM específicos para cada Product** para la predicción posterior de Issue.

Para garantizar una evaluación independiente de ambas etapas, se utilizó un holdout común de **5.086 observaciones**, pertenecientes simultáneamente al conjunto de prueba de DistilBERT y al conjunto de prueba utilizado para Issue.

En la primera etapa, DistilBERT alcanzó una **accuracy de 0,8748**, un **F1 macro de 0,7594** y un **F1 weighted de 0,8725** en este holdout.

En la evaluación completa end-to-end, utilizando el Product predicho para seleccionar el clasificador correspondiente de Issue, se obtuvo una **accuracy de 0,6258**, un **F1 macro de 0,3745** y un **F1 weighted de 0,6157**.

El escenario oracle, en el que se proporciona el Product real al clasificador de Issue, alcanzó una **accuracy de 0,6897**, un **F1 macro de 0,4801** y un **F1 weighted de 0,6803**. La comparación muestra una reducción absoluta de **0,1055 puntos de F1 macro**, equivalente a una caída relativa del **21,98 %**, cuando se pasa del escenario oracle al funcionamiento real end-to-end.

El análisis de propagación de errores confirma la dependencia entre ambas etapas. Cuando DistilBERT clasifica correctamente el Product, lo que ocurre en el **87,48 %** de las observaciones, Issue alcanza una accuracy de **0,7080** y un F1 macro de **0,4888**. En cambio, cuando Product se clasifica incorrectamente, la accuracy de Issue disminuye hasta **0,0518** y el F1 macro hasta **0,0257**.

Estos resultados muestran que la clasificación de Product constituye un punto crítico del sistema jerárquico: los errores producidos en la primera etapa se propagan hacia la segunda al dirigir la narrativa hacia un clasificador de Issue asociado a un Product incorrecto.

Finalmente, se implementó una función de inferencia que encapsula ambas etapas y permite recibir una nueva narrativa como entrada y devolver automáticamente las predicciones de **Product** e **Issue**, dejando preparado el pipeline para su posterior integración en una aplicación.

## Verificación de ejemplos sintéticos de la aplicación

La aplicación Streamlit incluye tres reclamaciones sintéticas creadas específicamente
para demostrar el funcionamiento del pipeline sobre textos nuevos.

A continuación se comprueba que estas narrativas no aparecen en el dataset utilizado
durante el desarrollo del proyecto.

In [38]:
ejemplos_sinteticos = {
    "Option 1": (
        "Last month I paid my credit card balance in full before the due date. "
        "However, my latest statement shows that I was charged a late payment fee. "
        "I contacted the card issuer and provided proof that the payment was made "
        "on time, but they refused to remove the fee and told me that I still have "
        "to pay it."
    ),

    "Option 2": (
        "I recently sold my home and the mortgage was paid off at closing. "
        "Three weeks later, I noticed that my mortgage servicer had withdrawn "
        "another monthly payment from my bank account. I contacted them to request "
        "a refund, but they told me the payment was still being processed and "
        "could not tell me when I would get my money back."
    ),

    "Option 3": (
        "I sent $1,200 to my sister through an online money transfer service. "
        "The money was immediately taken from my account, but five days later "
        "she still had not received it. The company keeps telling me that the "
        "transaction is pending and has not been able to explain where the money "
        "is or when it will arrive."
    ),
}

In [39]:
import pandas as pd

ruta_complaints = "../data/raw/complaints.csv"

columna_texto = "Consumer complaint narrative"

coincidencias_exactas = {
    nombre: 0 for nombre in ejemplos_sinteticos
}

for chunk in pd.read_csv(
    ruta_complaints,
    usecols=[columna_texto],
    chunksize=200_000,
    low_memory=False
):
    narrativas = chunk[columna_texto].dropna()

    for nombre, texto in ejemplos_sinteticos.items():
        coincidencias_exactas[nombre] += (narrativas == texto).sum()

coincidencias_exactas

{'Option 1': np.int64(0), 'Option 2': np.int64(0), 'Option 3': np.int64(0)}

In [40]:
import re

def normalizar_texto(texto):
    texto = str(texto).lower().strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto


# Normalizamos nuestros tres ejemplos
ejemplos_normalizados = {
    nombre: normalizar_texto(texto)
    for nombre, texto in ejemplos_sinteticos.items()
}

coincidencias_normalizadas = {
    nombre: 0 for nombre in ejemplos_sinteticos
}

for chunk in pd.read_csv(
    ruta_complaints,
    usecols=[columna_texto],
    chunksize=200_000,
    low_memory=False
):
    narrativas = (
        chunk[columna_texto]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    for nombre, texto in ejemplos_normalizados.items():
        coincidencias_normalizadas[nombre] += (
            narrativas == texto
        ).sum()

coincidencias_normalizadas

{'Option 1': np.int64(0), 'Option 2': np.int64(0), 'Option 3': np.int64(0)}

### Resultado de la verificación

Los tres ejemplos sintéticos utilizados en la aplicación Streamlit fueron contrastados
con el conjunto completo de narrativas del CFPB.

Se realizaron dos comprobaciones:

1. **Coincidencia exacta:** búsqueda literal de cada ejemplo dentro de la columna
   `Consumer complaint narrative`.
2. **Coincidencia normalizada:** comparación tras convertir los textos a minúsculas,
   eliminar espacios al inicio y al final y unificar espacios consecutivos.

En ambos casos se obtuvieron **0 coincidencias para los tres ejemplos**.

Por tanto, los textos utilizados en la demostración de la aplicación no aparecen en
el corpus original del CFPB. Esto permite utilizarlos como ejemplos sintéticos externos
al dataset para ilustrar cualitativamente el funcionamiento del pipeline sobre nuevas
narrativas. La capacidad de generalización del modelo se evalúa cuantitativamente de
forma independiente mediante los conjuntos de test y el holdout end-to-end.